In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# Add rebuild to path
sys.path.insert(0, str(Path.cwd() / "rebuild"))

# Set up output directories
OUTPUT_DIR = Path("./rebuild_outputs")
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Load results
df_results = pd.read_csv(TABLES_DIR / "results_all.csv")
df_summary = pd.read_csv(TABLES_DIR / "summary_statistics.csv")

print("✓ Loaded results from rebuild_outputs/")
print(f"  Total rows: {len(df_results)}")
print(f"  Models: {df_results['model'].unique().tolist()}")
print(f"  Mitigations: {df_results['mitigation'].unique().tolist()}")
print(f"  Perturbations: {df_results['perturbation'].unique().tolist()}")


# Summary Statistics Table

# Pivot to get means for each combination
summary_pivot = df_summary.pivot_table(
    index=['model', 'mitigation', 'perturbation'],
    values=['utility_mean', 'disparate_impact_mean', 'equal_opportunity_mean'],
    aggfunc='first'
).reset_index()

summary_pivot = summary_pivot.sort_values(['model', 'perturbation', 'mitigation'])

# Create interactive table
fig = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Model</b>', '<b>Mitigation</b>', '<b>Perturbation</b>', 
                '<b>Utility</b>', '<b>Disparate Impact</b>', '<b>Equal Opportunity</b>'],
        fill_color='lightgrey',
        align='left',
        font=dict(size=12)
    ),
    cells=dict(
        values=[
            summary_pivot['model'],
            summary_pivot['mitigation'],
            summary_pivot['perturbation'],
            summary_pivot['utility_mean'].round(3),
            summary_pivot['disparate_impact_mean'].round(3),
            summary_pivot['equal_opportunity_mean'].round(3),
        ],
        align='left',
        height=30
    )
)])

fig.update_layout(
    title="Summary: Mean Metrics Across Bootstrap Iterations",
    height=1200
)

try:
    fig.write_image(str(FIGURES_DIR / "01_summary_table.png"), scale=2, width=1200, height=1200)
    print(f"✓ Saved: {FIGURES_DIR / '01_summary_table.png'}")
except Exception as e:
    print(f"⚠ Could not save PNG (kaleido required): {e}")
    print(f"  Install with: pip install kaleido")

fig.show()


In [ ]:
# Heatmaps: Utility by Model, Mitigation, Perturbation

for model_name in sorted(df_summary['model'].unique()):
    df_model = df_summary[df_summary['model'] == model_name]
    
    # Pivot: rows=perturbation, cols=mitigation, values=utility_mean
    pivot_util = df_model.pivot_table(
        index='perturbation',
        columns='mitigation',
        values='utility_mean',
        aggfunc='first'
    )
    
    # Ensure consistent ordering
    pivot_util = pivot_util.reindex(
        index=['D0', 'D1A', 'D2A'],
        columns=['none', 'reweighting', 'smote', 'threshold_optimization']
    )
    
    fig = px.imshow(
        pivot_util,
        labels=dict(x="Mitigation", y="Perturbation", color="Utility"),
        text_auto='.3f',
        aspect='auto',
        color_continuous_scale='RdYlGn',
        zmin=0, zmax=1
    )
    
    fig.update_layout(
        title=f"{model_name}: PhysioNet Utility (ideal = 1.0)",
        height=400,
        width=700
    )
    
    fname = f"02_{model_name}_utility_heatmap.png"
    try:
        fig.write_image(str(FIGURES_DIR / fname), scale=2, width=700, height=400)
        print(f"✓ Saved: {FIGURES_DIR / fname}")
    except Exception as e:
        print(f"⚠ Could not save {fname}: {e}")
    
    fig.show()


# Heatmaps: Disparate Impact by Model, Mitigation, Perturbation

for model_name in sorted(df_summary['model'].unique()):
    df_model = df_summary[df_summary['model'] == model_name]
    
    # Pivot: rows=perturbation, cols=mitigation, values=disparate_impact_mean
    pivot_di = df_model.pivot_table(
        index='perturbation',
        columns='mitigation',
        values='disparate_impact_mean',
        aggfunc='first'
    )
    
    # Ensure consistent ordering
    pivot_di = pivot_di.reindex(
        index=['D0', 'D1A', 'D2A'],
        columns=['none', 'reweighting', 'smote', 'threshold_optimization']
    )
    
    fig = px.imshow(
        pivot_di,
        labels=dict(x="Mitigation", y="Perturbation", color="Disparate Impact"),
        text_auto='.3f',
        aspect='auto',
        color_continuous_scale='RdYlGn',
        zmin=0.5, zmax=1.5
    )
    
    fig.update_layout(
        title=f"{model_name}: Disparate Impact P(alarm|F)/P(alarm|M) (ideal = 1.0)",
        height=400,
        width=700
    )
    
    fname = f"03_{model_name}_disparate_impact_heatmap.png"
    try:
        fig.write_image(str(FIGURES_DIR / fname), scale=2, width=700, height=400)
        print(f"✓ Saved: {FIGURES_DIR / fname}")
    except Exception as e:
        print(f"⚠ Could not save {fname}: {e}")
    
    fig.show()


In [ ]:
import pandas as pd
import plotly.graph_objects as go

def make_model_table(results, model_name):

    dataset_map = {
        "D0": "Balanced",
        "D1A": "Row Removal",
        "D2A": "Missingness"
    }

    model_map = {
        "liu_glm": "GLM",
        "liu_xgboost": "XGBoost",
        "liu_rnn": "RNN"
    }

    mitigation_map = {
        "none": "Original",
        "reweighting": "Reweighting",
        "smote": "SMOTE",
        "threshold_optimization": "Fairness Penalty"
    }

    df = results.copy()

    df["Dataset"] = df["dataset_id"].map(dataset_map)
    df["Model"] = df["model"].map(model_map)
    df["Mitigation"] = df["mitigation"].map(mitigation_map)

    df = df.dropna(subset=["Dataset", "Model", "Mitigation"])
    df = df[df["Model"] == model_name]

    df = df[[
        "Mitigation", "Dataset",
        "overall_physionet_utility",
        "disparate_impact",
        "equal_opportunity"
    ]].rename(columns={
        "overall_physionet_utility": "Utility",
        "disparate_impact": "Disparate Impact",
        "equal_opportunity": "Equal Opportunity",
    }).round(3)

    mitigation_order = ["Original", "Reweighting", "SMOTE", "Fairness Penalty"]
    dataset_order = ["Balanced", "Row Removal", "Missingness"]

    df["Mitigation"] = pd.Categorical(df["Mitigation"], mitigation_order)
    df["Dataset"] = pd.Categorical(df["Dataset"], dataset_order)

    df = df.sort_values(["Dataset", "Mitigation"])

    # ── NEW grouping logic ───────────────────────
    rows = []

    for dataset in dataset_order:
        df_ds = df[df["Dataset"] == dataset]

        for mitigation in mitigation_order:
            df_mit = df_ds[df_ds["Mitigation"] == mitigation]

            for i, (_, r) in enumerate(df_mit.iterrows()):
                rows.append({
                    "Dataset": dataset if (mitigation == mitigation_order[0] and i == 0) else "",
                    "Mitigation": mitigation if i == 0 else "",
                    "Utility": r["Utility"],
                    "Disparate Impact": r["Disparate Impact"],
                    "Equal Opportunity": r["Equal Opportunity"],
                })

    table_df = pd.DataFrame(rows)

    fig = go.Figure(data=[go.Table(
        header=dict(
            values=list(table_df.columns),
            fill_color="lightgrey",
            align="left",
            font=dict(size=16)
        ),
        cells=dict(
            values=[table_df[c] for c in table_df.columns],
            align="left",
            height=35
        )
    )])

    fig.update_layout(
        title=f"{model_name} — Dataset-first Comparison",
        height=800
    )

    print("\n", model_name)
    print(table_df.to_string(index=False))

    return fig
figs = {}

for m in ["GLM", "RNN", "XGBoost"]:
    figs[m] = make_model_table(results, m)

for f in figs.values():
    f.show()



 GLM
    Dataset       Mitigation  Utility  Disparate Impact  Equal Opportunity
   Balanced         Original    0.147             0.877              0.096
                 Reweighting    0.130             0.871              0.119
                       SMOTE    0.145             0.884              0.096
            Fairness Penalty   -0.054             0.999              0.007
Row Removal         Original    0.305             0.913              0.025
                 Reweighting    0.304             0.899              0.028
                       SMOTE    0.327             0.899              0.006
            Fairness Penalty    0.137             1.139              0.023
Missingness         Original    0.148             0.878              0.104
                 Reweighting    0.127             0.873              0.127
                       SMOTE    0.145             0.881              0.104
            Fairness Penalty   -0.054             0.999              0.007

 RNN
    Dataset  

In [ ]:
# Heatmaps: Equal Opportunity by Model, Mitigation, Perturbation

for model_name in sorted(df_summary['model'].unique()):
    df_model = df_summary[df_summary['model'] == model_name]
    
    # Pivot: rows=perturbation, cols=mitigation, values=equal_opportunity_mean
    pivot_eo = df_model.pivot_table(
        index='perturbation',
        columns='mitigation',
        values='equal_opportunity_mean',
        aggfunc='first'
    )
    
    # Ensure consistent ordering
    pivot_eo = pivot_eo.reindex(
        index=['D0', 'D1A', 'D2A'],
        columns=['none', 'reweighting', 'smote', 'threshold_optimization']
    )
    
    fig = px.imshow(
        pivot_eo,
        labels=dict(x="Mitigation", y="Perturbation", color="Equal Opportunity"),
        text_auto='.3f',
        aspect='auto',
        color_continuous_scale='RdBu_r',
        zmid=0
    )
    
    fig.update_layout(
        title=f"{model_name}: Equal Opportunity TPR(F) - TPR(M) (ideal = 0.0)",
        height=400,
        width=700
    )
    
    fname = f"04_{model_name}_equal_opportunity_heatmap.png"
    try:
        fig.write_image(str(FIGURES_DIR / fname), scale=2, width=700, height=400)
        print(f"✓ Saved: {FIGURES_DIR / fname}")
    except Exception as e:
        print(f"⚠ Could not save {fname}: {e}")
    
    fig.show()


In [ ]:
# Box plots: Distribution of Utility Across Bootstrap Iterations

for model_name in sorted(df_results['model'].unique()):
    df_model = df_results[df_results['model'] == model_name]
    
    fig = px.box(
        df_model,
        x='mitigation',
        y='overall_physionet_utility',
        color='perturbation',
        facet_col='perturbation',
        labels={'overall_physionet_utility': 'Utility', 'mitigation': 'Mitigation'},
        title=f"{model_name}: Utility Distribution Across Bootstrap Iterations (N=5)",
        height=500,
        width=1000
    )
    
    fname = f"05_{model_name}_utility_boxplot.png"
    try:
        fig.write_image(str(FIGURES_DIR / fname), scale=2, width=1000, height=500)
        print(f"✓ Saved: {FIGURES_DIR / fname}")
    except Exception as e:
        print(f"⚠ Could not save {fname}: {e}")
    
    fig.show()

print("\n✓ All visualizations complete!")
print(f"Output directory: {FIGURES_DIR.resolve()}")
print("\nFiles saved:")
print("  - 01_summary_table.png")
print("  - 02_GRU_utility_heatmap.png")
print("  - 02_LogisticGLM_utility_heatmap.png")
print("  - 02_XGBoost_utility_heatmap.png")
print("  - 03_GRU_disparate_impact_heatmap.png")
print("  - 03_LogisticGLM_disparate_impact_heatmap.png")
print("  - 03_XGBoost_disparate_impact_heatmap.png")
print("  - 04_GRU_equal_opportunity_heatmap.png")
print("  - 04_LogisticGLM_equal_opportunity_heatmap.png")
print("  - 04_XGBoost_equal_opportunity_heatmap.png")
print("  - 05_GRU_utility_boxplot.png")
print("  - 05_LogisticGLM_utility_boxplot.png")
print("  - 05_XGBoost_utility_boxplot.png")
